In [60]:
import os
import numpy as np
import sympy as sp
import torch

from SymbolicMathematics.src.envs import set_cover_utils
from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules

from src.utils import to_cuda
from src.envs.sympy_utils import simplify
from src.envs.set_cover_utils import validate_set_cover_solution

from src.envs import char_sp
from src.envs.char_sp import CharSPEnvironment
from itertools import combinations

import json
import matplotlib.pyplot as plt

## Required Functions

In [61]:
def is_covering(matrix, selected_columns, rows):
    covered = set()
    for col in selected_columns:
        covered.update(np.where(matrix[:, col])[0])
    return len(covered) == rows

In [62]:
def set_cover_bruteforce(matrix):
    rows, cols = matrix.shape
    for num_columns in range(1, cols + 1):
        for selected_columns in combinations(range(cols), num_columns):
            if is_covering(matrix, selected_columns, rows):
                return selected_columns
    return None

In [63]:
def list_to_tokens(lst):
    return [str(c) for c in str(lst) if str(c) != ' ']

In [64]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    if len(token_sequence) == 0:
        return []

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    if len(list_of_lists) == 0:
        return []
    return list_of_lists[0]  # Return the final list of lists

In [65]:
seed = 1234
def gen_set_cover(max_universe_size=6):
    rng = np.random.RandomState(seed)
    # Generate a random binary matrix
    num_rows = rng.randint(3, max_universe_size + 1) #elements
    num_cols = rng.randint(2, 2*max_universe_size + 1) #subsets
    binary_matrix = np.random.choice([False, True], size=(num_rows, num_cols))  # Randomly generate binary matrix

    for i in range(num_rows):
        if not binary_matrix[i].any():
            binary_matrix[i, np.random.randint(0, num_cols)] = True
    for j in range(num_cols):
        if not binary_matrix[:, j].any():
            binary_matrix[np.random.randint(0, num_rows), j] = True

    # Remove duplicate rows
    binary_matrix = np.unique(binary_matrix, axis=1)

    # Find a set cover solution using brute force
    solution_indices = set_cover_bruteforce(binary_matrix)

    # Convert binary matrix and solution to sequences
    input_sequence = [list(np.where(column)[0]) for column in binary_matrix.T]
    # Map the solution subsets to their corresponding indices in the input sequence
    label_sequence = [subset for idx, subset in enumerate(input_sequence) if idx in solution_indices]

    input_sequence = list_to_tokens(input_sequence)
    label_sequence = list_to_tokens(label_sequence)

    return input_sequence, label_sequence

In [66]:
    seed = 125
    def gen_set_cover_2(max_universe_size=6):
        rng = np.random.RandomState(seed)
        universe_list = list(range(10))  # Default universe list from 0 to 20
        selected_elements = np.random.choice(universe_list, max_universe_size, replace=False)

        # Generate a random binary matrix
        num_rows = rng.randint(3, max_universe_size + 1) #elements
        num_cols = rng.randint(2, 2*max_universe_size + 1) #subsets
        binary_matrix = np.random.choice([False, True], size=(num_rows, num_cols))  # Randomly generate binary matrix

        for i in range(num_rows):
            if not binary_matrix[i].any():
                binary_matrix[i, np.random.randint(0, num_cols)] = True
        for j in range(num_cols):
            if not binary_matrix[:, j].any():
                binary_matrix[np.random.randint(0, num_rows), j] = True

        # Remove duplicate rows
        binary_matrix = np.unique(binary_matrix, axis=1)
        # Find a set cover solution using brute force
        solution_indices = set_cover_bruteforce(binary_matrix)

        # Convert binary matrix columns to sequences of selected elements
        input_sequence = []
        for column in binary_matrix.T:
            subset = [selected_elements[row] for row in np.where(column)[0]]
            input_sequence.append(subset)

        label_sequence = [subset for idx, subset in enumerate(input_sequence) if idx in solution_indices]

        input_sequence = list_to_tokens(input_sequence)
        label_sequence = list_to_tokens(label_sequence)


        return input_sequence, label_sequence

In [81]:
def validate_set_cover_solution(problem_set, solution_set, true_output):
    valid_solution_count = 0
    error_counts = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }
    coverage_percentage = 0

    # Check if the solution set is empty
    if not solution_set:
        error_counts["Wrong Format"] += 1
        return valid_solution_count, error_counts, coverage_percentage


    # Check if any subset in the solution set is empty
    for j, subset in enumerate(solution_set):
        if not subset:
            error_counts["Wrong Format"] += 1
            return valid_solution_count, error_counts, coverage_percentage

    # Check if all subsets in the solution set exist in the problem set
        if subset not in problem_set:
            error_counts["Subset not in problem set"] += 1
            return valid_solution_count, error_counts, coverage_percentage


    # Check if the solution set covers the entire problem set
    covered_tokens = set().union(*solution_set)
    problem_tokens = set(token for sublist in problem_set for token in sublist)
    if covered_tokens != problem_tokens:
        coverage_percentage = round(len(covered_tokens) / len(problem_tokens) * 100, 2)
        error_counts["Solution not covering problem set"] += 1

    else:
        if len(solution_set) > len(true_output):
            error_counts["Solution is not optimal"] += 1
        else:
            valid_solution_count += 1

    return valid_solution_count, error_counts, coverage_percentage

In [68]:
#Test
input1, label1 = gen_set_cover(6)
print("input sequence:", input1)
print("label sequence:", label1)

input sequence: ['[', '[', '3', ']', ',', '[', '2', ']', ',', '[', '1', ']', ',', '[', '0', ',', '3', ']', ']']
label sequence: ['[', '[', '2', ']', ',', '[', '1', ']', ',', '[', '0', ',', '3', ']', ']']


## Build environment / Reload model

In [43]:
# Reload Trained model
model_path1 = './Model1/first_train/2spv1sq7hj/checkpoint.pth'
assert os.path.isfile(model_path1)

model_path2 = './Model2/first_train/5r8wvsegfc/checkpoint.pth'
assert os.path.isfile(model_path2)


In [69]:
params = AttrDict({

    # environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    #'tasks': 'prim_fwd',
    'tasks': 'set_cover',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # model parameters
    'cpu': True,
    'emb_dim': 128,
    'n_enc_layers': 2, #it was 6
    'n_dec_layers': 2, #it was 6
    'n_heads': 2, #it was 8
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path2,

})

In [70]:
env = build_env(params)
#x = env.local_dict['x']

In [71]:
modules = build_modules(env, params)
encoder = modules['encoder']
decoder = modules['decoder']

In [72]:
problem_set_instance = [[1,2],[1,2,3],[2],[4]]
problem_instance_tokens = list_to_tokens(problem_set_instance)
print(problem_instance_tokens)

['[', '[', '1', ',', '2', ']', ',', '[', '1', ',', '2', ',', '3', ']', ',', '[', '2', ']', ',', '[', '4', ']', ']']


In [77]:
def test_model_on_gen_set_cover(num_samples=1000, beam_size=1):
    total_valid_answers = 0
    total_errors = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }

    # Generate and test samples
    for i in range(num_samples):
        input_sequence, label_sequence = gen_set_cover()

        # Encode the input sequence
        x1_prefix = input_sequence
        x1 = torch.LongTensor(
            [env.eos_index] +
            [env.word2id[w] for w in x1_prefix] +
            [env.eos_index]
        ).view(-1, 1)
        len1 = torch.LongTensor([len(x1)])
        # x1, len1 = to_cuda(x1, len1)  # Uncomment if you are using GPU
        with torch.no_grad():
            encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

        # Convert the input and label sequences to sets
        input_set = convert_token_sequence_to_list_of_lists(input_sequence)
        label_set = convert_token_sequence_to_list_of_lists(label_sequence)

        # Decode with beam size 1
        with torch.no_grad():
            _, _, beam = decoder.generate_beam(encoded, len1, beam_size=beam_size, length_penalty=1.0, early_stopping=1, max_len=200)
            assert len(beam) == 1
        hypotheses = beam[0].hyp
        assert len(hypotheses) == beam_size

        # Parse decoded hypothesis
        score, sent = hypotheses[0]
        ids = sent[1:].tolist()  # decoded token IDs
        tok = [env.id2word[wid] for wid in ids]  # convert to prefix

        try:
            hyp = convert_token_sequence_to_list_of_lists(tok)
        except:
            hyp = tok

        # Validate the solution
        valid_solution_count, error_counts, coverage_percentage = validate_set_cover_solution(input_set, hyp, label_set)

        # Check if the hypothesis is a valid solution
        if valid_solution_count > 0:
            total_valid_answers += 1

        # Accumulate error counts
        for error_type in total_errors:
            total_errors[error_type] += error_counts[error_type]

    # Calculate percentages
    valid_solution_percentage = (total_valid_answers / num_samples) * 100
    error_percentages = {error_type: (count / num_samples) * 100 for error_type, count in total_errors.items()}

    print(f"Accuracy with beam size 1: {valid_solution_percentage:.2f}%")
    for error_type, percentage in error_percentages.items():
        print(f"Percentage of {error_type}: {percentage:.2f}%")
        
    print("Check valid solution count", valid_solution_count )
    print("Check Errors", error_counts)

In [79]:
def test_model_on_gen_set_cover2(num_samples=1000, beam_size=1):
    total_valid_answers = 0
    total_errors = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }

    # Generate and test samples
    for i in range(num_samples):
        input_sequence, label_sequence = gen_set_cover_2()

        # Encode the input sequence
        x1_prefix = input_sequence
        x1 = torch.LongTensor(
            [env.eos_index] +
            [env.word2id[w] for w in x1_prefix] +
            [env.eos_index]
        ).view(-1, 1)
        len1 = torch.LongTensor([len(x1)])
        # x1, len1 = to_cuda(x1, len1)  # Uncomment if you are using GPU
        with torch.no_grad():
            encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

        # Convert the input and label sequences to sets
        input_set = convert_token_sequence_to_list_of_lists(input_sequence)
        label_set = convert_token_sequence_to_list_of_lists(label_sequence)

        # Decode with beam size 1
        with torch.no_grad():
            _, _, beam = decoder.generate_beam(encoded, len1, beam_size=beam_size, length_penalty=1.0, early_stopping=1, max_len=200)
            assert len(beam) == 1
        hypotheses = beam[0].hyp
        assert len(hypotheses) == beam_size

        # Parse decoded hypothesis
        score, sent = hypotheses[0]
        ids = sent[1:].tolist()  # decoded token IDs
        tok = [env.id2word[wid] for wid in ids]  # convert to prefix

        try:
            hyp = convert_token_sequence_to_list_of_lists(tok)
        except:
            hyp = tok

        # Validate the solution
        valid_solution_count, error_counts, coverage_percentage = validate_set_cover_solution(input_set, hyp, label_set)

        # Check if the hypothesis is a valid solution
        if valid_solution_count > 0:
            total_valid_answers += 1

        # Accumulate error counts
        for error_type in total_errors:
            total_errors[error_type] += error_counts[error_type]

    # Calculate percentages
    valid_solution_percentage = (total_valid_answers / num_samples) * 100
    error_percentages = {error_type: (count / num_samples) * 100 for error_type, count in total_errors.items()}
    

    print(f"Accuracy with beam size 1: {valid_solution_percentage:.2f}%")
    for error_type, percentage in error_percentages.items():
        print(f"Percentage of {error_type}: {percentage:.2f}%")


In [75]:
#Model 2
#Gen_set_cover
test_model_on_gen_set_cover(num_samples=1000)

Accuracy with beam size 1: 92.60%
Percentage of Wrong Format: 0.00%
Percentage of Subset not in problem set: 2.00%
Percentage of Solution not covering problem set: 4.30%
Percentage of Solution is not optimal: 1.10%


In [76]:
#Model 2
#Gen_set_cover 2
test_model_on_gen_set_cover2(num_samples=1000)

Accuracy with beam size 1: 92.20%
Percentage of Wrong Format: 0.00%
Percentage of Subset not in problem set: 3.00%
Percentage of Solution not covering problem set: 1.50%
Percentage of Solution is not optimal: 3.30%


In [56]:
#Model 1
#Gen_set_cover 
test_model_on_gen_set_cover(num_samples=1000)

Accuracy with beam size 1: 99.10%
Percentage of Wrong Format: 0.00%
Percentage of Subset not in problem set: 0.60%
Percentage of Solution not covering problem set: 0.10%
Percentage of Solution is not optimal: 0.20%


In [57]:
#Model 1
#Gen_set_cover 2
test_model_on_gen_set_cover2(num_samples=1000)

Accuracy with beam size 1: 0.00%
Percentage of Wrong Format: 10.60%
Percentage of Subset not in problem set: 89.30%
Percentage of Solution not covering problem set: 0.10%
Percentage of Solution is not optimal: 0.00%


In [82]:
#test
test_model_on_gen_set_cover2(num_samples=1000)

Check, valid 1
Check, counts {'Wrong Format': 0, 'Subset not in problem set': 0, 'Solution not covering problem set': 0, 'Solution is not optimal': 0}
Accuracy with beam size 1: 93.20%
Percentage of Wrong Format: 0.00%
Percentage of Subset not in problem set: 2.20%
Percentage of Solution not covering problem set: 0.90%
Percentage of Solution is not optimal: 3.70%
